In [0]:
from pyspark.sql.functions import col, trim, to_date, initcap, length, sum, when

In [0]:
# Use .read.table (Batch) instead of .readStream.table (Streaming)
temp_df = spark.read.table("practice.bronze.netflix_titles")
null_counts = temp_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in temp_df.columns
])

In [0]:
df = (spark.readStream.table("practice.bronze.netflix_titles"))
df_new = df.drop("_rescued_data")


In [0]:

# 1. First, filter out the "junk" (rows where date_added is a name or too short)
# A valid date like "September 25, 2021" is usually 15+ characters.
# Names like "Tony Tone" are short or don't follow the pattern.
df_date_filter = (df_new.filter(
    (col("date_added").isNotNull()) & 
    (length(trim(col("date_added"))) > 10) & # Filter out short names
    (col("date_added").rlike(".*[0-9]{4}.*"))) # Must contain a 4-digit year
)

# 2. Convert to Date
df_clean =( df_date_filter.withColumn("date_added_converted", 
    to_date(trim(col("date_added")), "MMMM d, yyyy"))\
        .dropna(subset=["show_id", "title", "date_added"])\
        .fillna({"director": "Unknown",
                "cast": "Not Available",
                "country": "Unknown",
                "rating": "Not Rated"}))
    


In [0]:
valid_ratings = [
    "PG-13", "TV-MA", "PG", "TV-14", "TV-PG", "TV-Y", 
    "TV-Y7", "R", "TV-G", "G", "NC-17", "NR", "TV-Y7-FV", "UR"
]
df_refined = df_clean.withColumn("rating", 
    when(col("rating").isin(valid_ratings), col("rating"))
    .otherwise("Not Rated") # This catches "74 min", etc., and standardizes them
)
df_final = df_refined.drop("date_added")

In [0]:
(df_final.writeStream
 .format("delta")
 .option("checkpointLocation", "/Volumes/practice/checkpoints/netflix_silver")
 .trigger(availableNow=True)
 .outputMode("append")
 .toTable("practice.silver.netflix_titles_cleaned"))